# 01 · Data Audit — AI-Enhanced Reorder Point & Inventory Management

**Course:** AI in Operations (MAIB, SP Jain) · **Exercise:** Session II — AI Applications in Reorder Point & Inventory Management

This notebook reproduces every number in Part A of `docs/IMPLEMENTATION_PLAN.md`:

1. Structure and consistency of the dataset
2. Demand drivers (seasonality, promotion, weather, epidemic)
3. **Target leakage** in `Price` / `Competitor Pricing`
4. **Lead times** recovered from the inventory ledger
5. Hand calculation of the slide formula and a Monte Carlo cross-check

Units throughout: demand in **units/day**, lead time in **days**.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

from inventory_data import build_panel, extract_lead_times, load_raw, validate

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)
df = load_raw(ROOT / "sales_data.csv")
validate(df)
panel = build_panel(df)
print(df.shape, "|", df.Date.min().date(), "->", df.Date.max().date(), "|", df.SKU.nunique(), "SKUs")
df.head()

(76000, 17) | 2022-01-01 -> 2024-01-30 | 100 SKUs


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand,SKU
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115,S001-P0001
1,2022-01-02,S001,P0001,Electronics,North,93,71,0,65.63,5,Snowy,0,73.66,Winter,0,84,S001-P0001
2,2022-01-03,S001,P0001,Electronics,North,274,142,229,68.55,15,Snowy,1,80.73,Winter,0,132,S001-P0001
3,2022-01-04,S001,P0001,Electronics,North,132,42,0,61.66,10,Snowy,0,54.88,Winter,0,67,S001-P0001
4,2022-01-05,S001,P0001,Electronics,North,319,129,0,59.56,25,Snowy,1,57.34,Winter,0,110,S001-P0001


## 1 · Structure and consistency
`validate()` already asserted: no missing values, no duplicate Date–SKU keys, every SKU has all 760 days,
weather constant per store-day and the epidemic flag constant per day. The checks below confirm the
remaining labels are coherent (unlike the original anirudhchauhan release, where they were random per row).

In [2]:
checks = pd.Series({
    "categories per SKU (max)": df.groupby("SKU").Category.nunique().max(),
    "regions per SKU (max)": df.groupby("SKU").Region.nunique().max(),
    "weather values per store-day (max)": df.groupby(["Date", "Store ID"])["Weather Condition"].nunique().max(),
    "negative numeric values": int((df.select_dtypes("number") < 0).sum().sum()),
})
display(checks.to_frame("value"))
display(pd.crosstab(df.Date.dt.month, df.Seasonality).rename_axis("month"))

,value
categories per SKU (max),1
regions per SKU (max),1
weather values per store-day (max),1
negative numeric values,0


Seasonality,Autumn,Spring,Summer,Winter
month,,,,
1,0,0,0,9200
2,0,0,0,5600
3,0,6200,0,0
4,0,6000,0,0
5,0,6200,0,0
6,0,0,6000,0
7,0,0,6200,0
8,0,0,6200,0
9,6000,0,0,0


In [3]:
ep = pd.Series(panel.epidemic, index=panel.dates)
runs = (ep.diff().fillna(ep.iloc[0]) != 0).cumsum()
episodes = ep.groupby(runs).agg(start=lambda s: s.index.min().date(), end=lambda s: s.index.max().date(), days="size", flag="first")
episodes = episodes[episodes.flag == 1].drop(columns="flag").reset_index(drop=True)
print(f"{len(episodes)} epidemic episodes, {episodes.days.sum()} days in total")
episodes

6 epidemic episodes, 152 days in total


,start,end,days
0,2022-04-23,2022-05-25,33
1,2022-06-29,2022-07-24,26
2,2023-01-29,2023-02-28,31
3,2023-04-12,2023-05-12,31
4,2023-10-07,2023-10-24,18
5,2023-12-27,2024-01-08,13


## 2 · Demand drivers
Effects are estimated with a log-demand regression with SKU fixed effects, so each coefficient is an average
multiplicative effect *within* a SKU. Price is deliberately left out here (see section 3).

In [4]:
d = df.rename(columns={"Weather Condition": "weather"}).assign(month=df.Date.dt.month, dow=df.Date.dt.dayofweek, logD=np.log(df.Demand))
fe = smf.ols("logD ~ C(SKU) + Promotion + Epidemic + C(weather, Treatment('Cloudy')) + C(month)", data=d).fit()
effects = fe.params.filter(regex="^(Promotion|Epidemic|C\\(weather|C\\(month)")
table = pd.DataFrame({"log coef": effects, "multiplier": np.exp(effects), "p-value": fe.pvalues[effects.index]}).round(4)
print(f"R² = {fe.rsquared:.3f}")
table

R² = 0.407


,log coef,multiplier,p-value
"C(weather, Treatment('Cloudy'))[T.Rainy]",-0.0988,0.9059,0.0000
"C(weather, Treatment('Cloudy'))[T.Snowy]",-0.1015,0.9035,0.0000
"C(weather, Treatment('Cloudy'))[T.Sunny]",0.0995,1.1046,0.0000
C(month)[T.2],0.0163,1.0164,0.0234
C(month)[T.3],0.0226,1.0229,0.0013
C(month)[T.4],0.0109,1.0109,0.1339
C(month)[T.5],0.0185,1.0187,0.0114
C(month)[T.6],0.0526,1.0541,0.0000
C(month)[T.7],0.0632,1.0652,0.0000
C(month)[T.8],0.0683,1.0706,0.0000


In [5]:
daily = df.groupby("Date").Demand.sum()
print("month ANOVA     p =", stats.f_oneway(*[v for _, v in daily.groupby(daily.index.month)]).pvalue)
print("weekday ANOVA   p =", stats.f_oneway(*[v for _, v in daily.groupby(daily.index.dayofweek)]).pvalue)
lag_ac = df.groupby("SKU").Demand.apply(lambda s: pd.Series({f"lag{k}": s.autocorr(k) for k in (1, 7, 365)})).unstack()
print("per-SKU autocorrelation (mean):", lag_ac.mean().round(3).to_dict())
print("Discount × Promotion:")
pd.crosstab(df.Discount, df.Promotion)

month ANOVA     p = 2.941109576729222e-44
weekday ANOVA   p = 0.9955735630147237
per-SKU autocorrelation (mean): {'lag1': 0.239, 'lag7': 0.166, 'lag365': 0.049}
Discount × Promotion:


Promotion,0,1
Discount,,
0,17126,0
5,16918,0
10,16956,6342
15,0,6245
20,0,6191
25,0,6222


**Reading:** epidemic ≈ −44 %, promotion ≈ +30 %, sunny +10 % / rainy & snowy −10 %, summer months +5–7 %,
no weekday effect. Discount ≥ 15 % only ever occurs during a promotion, and every promotion carries ≥ 10 %,
so discount adds nothing beyond the promotion flag.

## 3 · Target leakage in Price and Competitor Pricing

In [6]:
within = df.groupby("SKU").apply(lambda g: pd.Series({
    "corr(Price, Demand)": g.Price.corr(g.Demand),
    "corr(Price_t, Demand_t+1)": g.Price.corr(g.Demand.shift(-1)),
}), include_groups=False).mean()
print(within.round(3).to_dict())
print("corr(Price, Competitor Pricing) =", round(df.Price.corr(df["Competitor Pricing"]), 4))
with_price = smf.ols("logD ~ C(SKU) + Promotion + Epidemic + C(weather, Treatment('Cloudy')) + C(month) + np.log(Price)", data=d).fit()
print(f"R² without price {fe.rsquared:.3f}  ->  with log(Price) {with_price.rsquared:.3f}")
pd.DataFrame({"without price": fe.params, "with log(Price)": with_price.params}).filter(
    regex="^(Promotion|Epidemic|np.log)", axis=0).round(3)

{'corr(Price, Demand)': 0.797, 'corr(Price_t, Demand_t+1)': 0.224}
corr(Price, Competitor Pricing) = 0.9766


R² without price 0.407  ->  with log(Price) 0.816


,without price,with log(Price)
Epidemic,-0.575,-0.192
Promotion,0.260,0.409
np.log(Price),NaN,2.670


Adding same-day price doubles R² (≈0.41 → ≈0.82), distorts the real effects (the epidemic effect collapses
from −0.58 to −0.19) and implies a **positive** price "elasticity" of ≈ +2.7 — higher prices "causing" higher
demand, which is economically backwards. Price tracks *same-day* demand (r ≈ 0.80) far more than next-day
demand (r ≈ 0.22, which is just 0.80 × demand's own lag-1 autocorrelation of 0.24).
Price is therefore generated *from* the day's demand: using it (or its 0.98-correlated twin, Competitor Pricing)
as a feature would leak the answer. **Both are excluded from every forecasting model.** The SKU's median price
is kept only as a cost parameter.

## 4 · Lead times from the inventory ledger
The ledger closes as `inventory[t+1] = inventory[t] − sold[t] + receipt[t]`. Each order is matched to the
first later receipt of exactly the same quantity (`extract_lead_times`).

In [7]:
records, unmatched_total, n_orders = [], 0, 0
for i, sku in enumerate(panel.skus):
    matches, unmatched = extract_lead_times(panel.inventory[i], panel.units_sold[i], panel.units_ordered[i])
    unmatched_total += unmatched
    n_orders += len(matches) + unmatched
    records += [(sku, panel.sku_store[i], lt) for _, lt, _ in matches]
lt = pd.DataFrame(records, columns=["sku", "store", "L"])
print(f"matched {len(lt)} of {n_orders} orders ({len(lt) / n_orders:.1%}); unmatched {unmatched_total}")
print("lead-time distribution (share):", lt.L.value_counts(normalize=True).sort_index().round(3).to_dict())
print("within-SKU sd (max):", lt.groupby("sku").L.std().max())
lt.groupby("store").L.agg(["mean", "std", "count"]).round(4)

matched 28831 of 28893 orders (99.8%); unmatched 62
lead-time distribution (share): {1: 0.56, 2: 0.191, 3: 0.149, 4: 0.101}
within-SKU sd (max): 0.0


,mean,std,count
store,,,
S001,1.5897,0.9727,6585
S002,2.0699,1.0984,4924
S003,1.8896,1.0995,5416
S004,1.7600,1.0094,5928
S005,1.7258,0.9436,5978


Lead time is **constant within each SKU** (1–4 days) but varies across a store's supplier lanes.
Modelling decision: `L` = the SKU's lead time; `σ_L` = its store's pooled sd (documented assumption,
adjustable on the dashboard). Using the within-SKU sd (0) would silently switch off half of the slide's formula.

## 5 · Hand calculation of the slide formula
$$SS = Z\sqrt{\sigma_D^2 L + \bar D^2 \sigma_L^2}, \qquad ROP = \bar D L + SS$$

In [8]:
i = panel.skus.index("S001-P0001")
y = panel.demand[i]
pool = lt[lt.store == "S001"].L.to_numpy()
D_bar, sigma_D = y.mean(), y.std(ddof=1)
L = float(lt[lt.sku == "S001-P0001"].L.mean())
sigma_L = pool.std(ddof=1)
Z = stats.norm.ppf(0.95)
term_demand, term_lead = sigma_D**2 * L, D_bar**2 * sigma_L**2
SS = Z * np.sqrt(term_demand + term_lead)
ROP = D_bar * L + SS
hand = pd.Series({
    "D̄ (units/day)": D_bar, "σ_D (units/day)": sigma_D, "L (days)": L, "σ_L (days)": sigma_L, "Z": Z,
    "σ_D²·L": term_demand, "D̄²·σ_L²": term_lead, "√(sum)": np.sqrt(term_demand + term_lead),
    "Safety stock (units)": SS, "ROP (units)": ROP, "lead-time share of variance": term_lead / (term_demand + term_lead),
}).round(4)
assert abs(SS - 169.95) < 0.01 and abs(ROP - 355.58) < 0.01
hand.to_frame("S001-P0001")

,S001-P0001
D̄ (units/day),92.8145
σ_D (units/day),35.5362
L (days),2.0000
σ_L (days),0.9727
Z,1.6449
σ_D²·L,2525.6465
D̄²·σ_L²,8150.4330
√(sum),103.3251
Safety stock (units),169.9547
ROP (units),355.5836


### Monte Carlo cross-check
Lead times are drawn from S001's empirical pool and daily demand is bootstrapped from history. Demand during
lead time is a random sum, whose variance is exactly $\sigma_D^2 E[L] + \bar D^2 Var(L)$ — the expression under
the slide's square root. The simulation should reproduce that variance, and reveal the tail the Normal
approximation misses.

In [9]:
rng = np.random.default_rng(7)
N = 200_000
vals, counts = np.unique(pool, return_counts=True)
p = counts / counts.sum()
L_draw = rng.choice(vals, size=N, p=p)
daily_draws = rng.choice(y, size=(N, vals.max()))
ddlt = (daily_draws * (np.arange(vals.max())[None, :] < L_draw[:, None])).sum(axis=1)
m_L, v_L = (vals * p).sum(), ((vals - (vals * p).sum()) ** 2 * p).sum()
sd_theory = np.sqrt(sigma_D**2 * m_L + D_bar**2 * v_L)
ss_formula = Z * sd_theory
ss_mc = np.quantile(ddlt, 0.95) - ddlt.mean()
pd.Series({
    "MC mean": ddlt.mean(), "theory mean D̄·E[L]": D_bar * m_L,
    "MC sd": ddlt.std(), "theory sd (formula)": sd_theory,
    "skewness": stats.skew(ddlt),
    "SS formula": ss_formula, "SS Monte Carlo (95% quantile)": ss_mc,
    "P(stockout) at formula ROP": np.mean(ddlt > D_bar * m_L + ss_formula),
}).round(4).to_frame("value")

,value
MC mean,147.5886
theory mean D̄·E[L],147.5447
MC sd,100.6569
theory sd (formula),100.7803
skewness,1.4094
SS formula,165.7689
SS Monte Carlo (95% quantile),224.4114
P(stockout) at formula ROP,0.0949


The formula's variance is exact, but demand during lead time is right-skewed (4-day lead times), so the
Normal 95 % quantile under-protects: about **9.5 % stockouts instead of 5 %**. This is why the dashboard
sizes safety stock with Monte Carlo on top of the slide's formula.